In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent / 'src'))

from kafka import KafkaConsumer, TopicPartition
from models import Ride, ride_deserializer, green_ride_deserializer
import psycopg2
from datetime import datetime
import time

server = 'localhost:9092'
topic_name = 'rides'
green_topic_name = 'green-trips'

In [ ]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [2]:
green_consumer = KafkaConsumer(
    green_topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='green-trips-console',
    value_deserializer=green_ride_deserializer
)

In [3]:
conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True

cur = conn.cursor()

In [ ]:
print(f"Listening to {topic_name} and writing to PostgreSQL")

count = 0
t0 = time.time()

consumer.subscribe(['rides'])

while not consumer.assignment():
    consumer.poll(timeout_ms=100)

end_offsets = consumer.end_offsets(consumer.assignment())

for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute("""
            INSERT INTO processed_events
                (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
                VALUES (%s, %s, %s, %s, %s)""",
                (ride.PULocationID, ride.DOLocationID,
                 ride.trip_distance, ride.total_amount, pickup_dt)
                )
    count += 1
    if count % 1000 == 0:
        print(f"Inserted {count} rows")
        conn.commit()

    tp = TopicPartition(message.topic, message.partition)
    if consumer.position(tp) >= end_offsets[tp]:
        break

conn.commit()
t1 = time.time()

print(f"Took {t1 - t0} seconds to insert {count} rows")

consumer.close()
cur.close()
conn.close()

In [4]:
print(f"Listening to {green_topic_name} and writing to PostgreSQL")

count = 0
t0 = time.time()

green_consumer.subscribe(['green-trips'])

while not green_consumer.assignment():
    green_consumer.poll(timeout_ms=100)

end_offsets = green_consumer.end_offsets(green_consumer.assignment())

for message in green_consumer:
    ride = message.value
    cur.execute("""
            INSERT INTO green_taxi_events
                (lpep_pickup_datetime, lpep_dropoff_datetime, PULocationID, DOLocationID, passenger_count, trip_distance, tip_amount, total_amount)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
                (ride['lpep_pickup_datetime'], ride['lpep_dropoff_datetime'], ride['PULocationID'], ride['DOLocationID'],
                 ride['passenger_count'], ride['trip_distance'], ride['tip_amount'], ride['total_amount'])
                )
    count += 1
    if count % 1000 == 0:
        print(f"Inserted {count} rows")
        conn.commit()

    tp = TopicPartition(message.topic, message.partition)
    if green_consumer.position(tp) >= end_offsets[tp]:
        break

conn.commit()
t1 = time.time()

print(f"Took {t1 - t0} seconds to insert {count} rows")

green_consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL
Inserted 1000 rows
Inserted 2000 rows
Inserted 3000 rows
Inserted 4000 rows
Inserted 5000 rows
Inserted 6000 rows
Inserted 7000 rows
Inserted 8000 rows
Inserted 9000 rows
Inserted 10000 rows
Inserted 11000 rows
Inserted 12000 rows
Inserted 13000 rows
Inserted 14000 rows
Inserted 15000 rows
Inserted 16000 rows
Inserted 17000 rows
Inserted 18000 rows
Inserted 19000 rows
Inserted 20000 rows
Inserted 21000 rows
Inserted 22000 rows
Inserted 23000 rows
Inserted 24000 rows
Inserted 25000 rows
Inserted 26000 rows
Inserted 27000 rows
Inserted 28000 rows
Inserted 29000 rows
Inserted 30000 rows
Inserted 31000 rows
Inserted 32000 rows
Inserted 33000 rows
Inserted 34000 rows
Inserted 35000 rows
Inserted 36000 rows
Inserted 37000 rows
Inserted 38000 rows
Inserted 39000 rows
Inserted 40000 rows
Inserted 41000 rows
Inserted 42000 rows
Inserted 43000 rows
Inserted 44000 rows
Inserted 45000 rows
Inserted 46000 rows
Inserted 47000 rows
Inserted 48000 row